#Severity Estimation Model

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

In [3]:
# 1. Load Data
df = pd.read_csv('synthetic_crime_severity_real_zones.csv')

# 2. Create Classes (Low/Medium/High) from the Score
# This allows us to calculate F1 Score and Accuracy
bins = [-1, 33, 66, 101]
labels = ['Low', 'Medium', 'High']
df['Severity_Class'] = pd.cut(df['Harm_Score'], bins=bins, labels=labels)

# 3. Preprocessing
le_enc = LabelEncoder()
df['crime_code'] = le_enc.fit_transform(df['crime'])
df['zone_code'] = le_enc.fit_transform(df['Patrol_Zone_DS'])
df['Hour'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.hour.fillna(0)

X = df[['crime_code', 'Hour', 'zone_code', 'victim_age']]
y = le_enc.fit_transform(df['Severity_Class']) # Encode Target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 4. Compare Models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier()
}

print(f"{'Model':<20} | {'Accuracy':<10} | {'F1 Score':<10}")
print("-" * 45)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"{name:<20} | {acc:.4f}     | {f1:.4f}")

Model                | Accuracy   | F1 Score  
---------------------------------------------
Random Forest        | 0.9482     | 0.9483
Gradient Boosting    | 0.9503     | 0.9504
SVM                  | 0.8571     | 0.8538
KNN                  | 0.7598     | 0.7616
